In [1]:
import pandas as pd

df = pd.read_csv("data/Dry_Bean_Dataset.csv", encoding="utf-8-sig")
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum().sum())
print(df.iloc[:, -1].value_counts())

(13611, 17)
['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4', 'Class']
0
Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64


In [2]:
print(df.duplicated().sum())
print(df.dtypes)

68
Area                 int64
Perimeter          float64
MajorAxisLength    float64
MinorAxisLength    float64
AspectRation       float64
Eccentricity       float64
ConvexArea           int64
EquivDiameter      float64
Extent             float64
Solidity           float64
roundness          float64
Compactness        float64
ShapeFactor1       float64
ShapeFactor2       float64
ShapeFactor3       float64
ShapeFactor4       float64
Class                  str
dtype: object


## Dataset Description

**Dataset:** Dry Bean Dataset, UCI Machine Learning Repository

**Source link:** https://archive.ics.uci.edu/dataset/602/dry+bean+dataset

**Task:** Multi-class classification of dry bean varieties

### Background
The Dry Bean dataset was created by Koklu and Özkan (2020) to classify dry bean varieties
using computer vision. Images of 13,611 bean grains from 7 registered varieties were taken
with a high-resolution camera. Each image went through segmentation and feature extraction,
giving 16 numeric features per grain: 12 dimension features and 4 shape-form features.
Several of these varieties look alike, so the task in this project is to predict the variety
of a bean from these measurements, which makes it a 7-class classification problem.

### Size
| Property | Value |
|---|---|
| Instances (rows) | 13,611 |
| Features | 16 |
| Target column | `Class` |
| Number of classes | 7 |
| Missing values | 0 |
| Duplicate rows | 68 (removed during preprocessing, leaving 13,543 rows) |

### Features
All 16 features are numeric (2 integer, 14 decimal): Area, Perimeter, MajorAxisLength,
MinorAxisLength, AspectRation, Eccentricity, ConvexArea, EquivDiameter, Extent, Solidity,
roundness, Compactness, ShapeFactor1, ShapeFactor2, ShapeFactor3, ShapeFactor4.

The 16 features are the size and shape measurements of each bean. The UCI page describes
them as 12 dimension features and 4 shape-form features. The data file names one column
`AspectRation` (the UCI page calls it AspectRatio), and I keep the file's name.

### Target classes
| Class | Before cleaning | After cleaning | Share (after) |
|---|---|---|---|
| DERMASON | 3,546 | 3,546 | 26.2% |
| SIRA | 2,636 | 2,636 | 19.5% |
| SEKER | 2,027 | 2,027 | 15.0% |
| HOROZ | 1,928 | 1,860 | 13.7% |
| CALI | 1,630 | 1,630 | 12.0% |
| BARBUNYA | 1,322 | 1,322 | 9.8% |
| BOMBAY | 522 | 522 | 3.9% |
| **Total** | **13,611** | **13,543** | |

### Class imbalance
The classes are imbalanced. DERMASON is the largest (26.2% of the data) and BOMBAY the
smallest (3.9%), about 6.8 times fewer rows. Because of this, accuracy alone can be misleading,
so AUC, F1 and MCC are also reported. The train/test split is stratified so every class
appears in both sets in the same proportions.

### Data quality
- No missing values.
- 68 exact duplicate rows were found and removed before the split, so identical rows
  cannot appear in both train and test. All 68 belonged to the HOROZ class
  (1,928 to 1,860 rows); the other classes were unchanged.

**Credit:** Koklu, M. and Özkan, I. A. (2020). Multiclass classification of dry beans using
computer vision and machine learning techniques. Computers and Electronics in Agriculture.
Dataset: UCI Machine Learning Repository, https://doi.org/10.24432/C50S4B (CC BY 4.0).

## Preprocessing

### Cleaning step 1: duplicates

In [3]:
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
rows_after = len(df)

print("Rows before:", rows_before)
print("Rows after :", rows_after)
print("Removed    :", rows_before - rows_after)
print("Duplicates left:", df.duplicated().sum())
print(df["Class"].value_counts())

Rows before: 13611
Rows after : 13543
Removed    : 68
Duplicates left: 0
Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1860
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64


In [4]:
print(df.drop(columns="Class").duplicated().sum())

0


68 exact duplicate rows were found and dropped, leaving 13,543 rows.

### Target classes
| Class | Before cleaning | After cleaning | Share (after) |
|---|---|---|---|
| DERMASON | 3,546 | 3,546 | 26.2% |
| SIRA | 2,636 | 2,636 | 19.5% |
| SEKER | 2,027 | 2,027 | 15.0% |
| HOROZ | 1,928 | 1,860 | 13.7% |
| CALI | 1,630 | 1,630 | 12.0% |
| BARBUNYA | 1,322 | 1,322 | 9.8% |
| BOMBAY | 522 | 522 | 3.9% |
| **Total** | **13,611** | **13,543** | |

### Cleaning step 2: encode the target and split the data
The target `Class` is converted to numbers with `LabelEncoder`. The data is split 80/20 with
stratification (so every class keeps its share in both parts) and a fixed random state.
Feature scaling is done inside the model pipelines, so it is fitted on training data only.

In [5]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

RANDOM_STATE = 7   # use your own number, but keep it the same everywhere

X = df.drop(columns="Class")
le = LabelEncoder()
y = le.fit_transform(df["Class"])
print(dict(zip(le.classes_, range(len(le.classes_)))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, shuffle=True, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)
print(pd.Series(le.inverse_transform(y_train)).value_counts(normalize=True).round(3))
print(pd.Series(le.inverse_transform(y_test)).value_counts(normalize=True).round(3))

{'BARBUNYA': 0, 'BOMBAY': 1, 'CALI': 2, 'DERMASON': 3, 'HOROZ': 4, 'SEKER': 5, 'SIRA': 6}
(10834, 16) (2709, 16)
DERMASON    0.262
SIRA        0.195
SEKER       0.150
HOROZ       0.137
CALI        0.120
BARBUNYA    0.098
BOMBAY      0.039
Name: proportion, dtype: float64
DERMASON    0.262
SIRA        0.195
SEKER       0.150
HOROZ       0.137
CALI        0.120
BARBUNYA    0.098
BOMBAY      0.038
Name: proportion, dtype: float64


In [6]:
import os
os.makedirs("model", exist_ok=True)

test_df = X_test.copy()
test_df["Class"] = le.inverse_transform(y_test)
test_df.to_csv("test_data.csv", index=False)
print(test_df.shape)

(2709, 17)


## Model training
Five classifiers are trained on the same training split. Logistic Regression, KNN and SVM use
a `StandardScaler` inside a pipeline because the features are on very different scales
(for example `Area` is in the tens of thousands while `Extent` is below 1).

In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": make_pipeline(StandardScaler(), SVC(probability=True, random_state=RANDOM_STATE)),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    print("trained:", name)

trained: Logistic Regression
trained: Decision Tree
trained: KNN
trained: Naive Bayes


c:\Users\hp\OneDrive\Desktop\classify-compare-deploy\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


trained: Support Vector Machine


## Evaluation
All six metrics are computed on the 20% test split. Because there are 7 classes, AUC is
one-vs-rest with macro averaging, and precision, recall and F1 use weighted averaging.

In [8]:
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

rows = []
for name, m in models.items():
    pred = m.predict(X_test)
    proba = m.predict_proba(X_test)
    rows.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, pred),
        "AUC": roc_auc_score(y_test, proba, multi_class="ovr", average="macro"),
        "Precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1": f1_score(y_test, pred, average="weighted", zero_division=0),
        "MCC": matthews_corrcoef(y_test, pred),
    })

results = pd.DataFrame(rows).set_index("ML Model Name").round(4)
results

,Accuracy,AUC,Precision,Recall,F1,MCC
ML Model Name,,,,,,
Logistic Regression,0.9232,0.9945,0.9233,0.9232,0.9232,0.9072
Decision Tree,0.9033,0.9491,0.9031,0.9033,0.9031,0.8831
KNN,0.9214,0.9855,0.9215,0.9214,0.9214,0.9049
Naive Bayes,0.7619,0.9665,0.7614,0.7619,0.7599,0.7125
Support Vector Machine,0.9288,0.9956,0.9290,0.9288,0.9288,0.9138


## Save trained models
The pipelines (scaler plus model) are saved with joblib, together with the label encoder.
The Streamlit app loads these files and never retrains.

In [9]:
import joblib

files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib",
}
for name, f in files.items():
    joblib.dump(models[name], f"model/{f}")
joblib.dump(le, "model/label_encoder.joblib")
print(sorted(os.listdir("model")))

['decision_tree.joblib', 'knn.joblib', 'label_encoder.joblib', 'logistic_regression.joblib', 'naive_bayes.joblib', 'svm.joblib']
